There are some global bounds on what variables can realistically be (e.g. even under climate change, mean temperatures shouldn't be 500K anywhere globally!) but we also want to evaluate whether output is within plausible ranges for a given location and time of year. E.g., lots of places are 25C, but 25C in Antarctica in the winter should raise eyebrows. Here we construct plausible ranges for each day of year, based on the variability in the coarse GCM modeled changes relative to the historical, and variability in observations.

In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import icechunk
import matplotlib.pyplot as plt
import zarr
import pandas as pd

from srm import catalog
from srm.qaqc import calculate_reasonable_bounds_doy
from srm.utils import open_icechunk

In [2]:
#import coiled

#cluster = coiled.Cluster(n_workers=20, region="us-west-2")  # same region as the S3 bucket
#client = cluster.get_client()

## Get data

In [3]:
path_output = (
    "s3://carbonplan-srm/output/production/CESM2-WACCM-ERA5-global.icechunk"
    # "s3://carbonplan-scratch/srm/output/qa/CESM2-WACCM-ERA5-lat-35.0to-22.0_lon16.0to33.0.icechunk/"
)
path_qaqc = "/scratch/synced/qa_plots/plausible_value_check/"
branch = "v0.10.0"  # "qa-qc-downscaling-step/vtest01"  #

spatial_subset = False

In [4]:
def print_frac_implausible(
    too_high, too_low, variable=None, scenario=None, ensemble_member=None, log_path=None
):
    frac_area_too_high = (too_high > 0).mean(dim=["lat", "lon"]).values
    frac_area_too_low = (too_low > 0).mean(dim=["lat", "lon"]).values
    print("Fraction of area too high: " + str(frac_area_too_high))
    print("Fraction of area too low: " + str(frac_area_too_low))

    if log_path is not None:
        df = pd.DataFrame(
            [
                {
                    "variable": variable,
                    "scenario": scenario,
                    "ensemble_member": ensemble_member,
                    "metric": "frac_area_too_high",
                    "value": frac_area_too_high,
                },
                {
                    "variable": variable,
                    "scenario": scenario,
                    "ensemble_member": ensemble_member,
                    "metric": "frac_area_too_low",
                    "value": frac_area_too_low,
                },
            ]
        )
        df.to_csv(log_path, mode="w", index=False)

# Loop through a bunch of scenarios

In [5]:
var_list = ["tas", "tasmax", "tasmin", 
            "tas", "tasmax", "tasmin",
            "tas", "tasmax", "tasmin"
           ]
scenario_list = [
    "historical",
    "historical",
    "historical",
    "g6_1p5k",
    "g6_1p5k",
    "g6_1p5k",
    "ssp245",
    "ssp245",
    "ssp245",
]
ens_list = ["r3i1p1f1", "001", "001", 
            "003", "003", "003", 
            "008", "008", "008"
           ]

In [7]:
extra_buffer_K = 5

In [8]:
def plot_map_with_boundaries(ds):
    fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="darkgray")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="darkgray")

    (ds).plot(ax=ax, transform=ccrs.PlateCarree(), label="Outside bounds")

    ax.gridlines(draw_labels=True, color="darkgray")

In [ ]:
for i, var in enumerate(var_list):
    scenario = scenario_list[i]
    ens = ens_list[i]

    print("######################################################")
    print(var)
    print(scenario)
    print(ens)
    print("######################################################")

    debiased_downscaled = open_icechunk(
        path=path_output,
        branch=branch,
        group=scenario + "/" + var + "/" + ens,
    )[var]

    obs_fine = catalog.get("ERA5").to_xarray()[var]
    obs_fine = obs_fine.where(obs_fine["time.year"] >= 1978, drop=True)

    raw_scenario = (
        catalog.get("CESM2-WACCM").to_xarray()[scenario][var].sel(ensemble_member=ens)
    )
    raw_historical = catalog.get("CESM2-WACCM").to_xarray()["historical"][var]

    if spatial_subset:
        obs_fine_subset = obs_fine.sel(lat=slice(-35, -22), lon=slice(16, 33))
        raw_scenario_subset = raw_scenario.sel(lat=slice(-35, -22), lon=slice(16, 33)).load()
        raw_historical_subset = raw_historical.sel(
            lat=slice(-35, -22), lon=slice(16, 33)
        ).load()
    else:
        obs_fine_subset = obs_fine
        raw_scenario_subset = raw_scenario
        raw_historical_subset = raw_historical

    low_bound, high_bound = calculate_reasonable_bounds_doy(
        raw_scenario_subset=raw_scenario_subset,
        raw_historical_subset=raw_historical_subset,
        obs_fine_subset=obs_fine_subset,
    )

    high_bound_generous = high_bound + extra_buffer_K
    low_bound_generous = low_bound - extra_buffer_K
    if extra_buffer_K>0:
        prefix="5Kbuffer_"
    else:
        prefix=""

    # Very slow step
    doy = debiased_downscaled["time.dayofyear"]
    too_high = (
        (debiased_downscaled > high_bound_generous.sel(dayofyear=doy))
        .mean(dim="time")
        .compute()
    )
    too_low = (
        (debiased_downscaled < low_bound_generous.sel(dayofyear=doy)).mean(dim="time").compute()
    )

    print_frac_implausible(
        too_high,
        too_low,
        variable=var,
        scenario=scenario,
        ensemble_member=ens,
        log_path=path_qaqc
        + "area_out_of_range_frac_"
        + prefix
        + var
        + "_"
        + scenario
        + "_"
        + ens
        + ".csv",
    )
    print(too_high.where(too_high > 0, drop=True))
    print(too_low.where(too_high > 0, drop=True))

    plot_map_with_boundaries(ds=(too_high + too_low) > 0)
    plt.savefig(path_qaqc + "map_out_of_range_" + prefix+var + "_" + scenario + "_" + ens + ".png")

    plt.figure()
    (too_high + too_low).plot()
    plt.savefig(path_qaqc + "map_out_of_range_no_boundaries" + prefix+var + "_" + scenario + "_" + ens + ".png")

    #plot_map_with_boundaries(ds=(too_low) > 0)
    #plt.savefig(path_qaqc + "map_min_out_of_range_" +prefix+ var + "_" + scenario + "_" + ens + ".png")

    #plot_map_with_boundaries(ds=(too_high) > 0)
    #plt.savefig(path_qaqc + "map_max_out_of_range_" +prefix+ var + "_" + scenario + "_" + ens + ".png")

######################################################
tasmax
ssp245
008
######################################################
